# Duplicate checking and Leakage checking

## PHẦN 1: Duplicate checking

In [17]:
from pathlib import Path
import hashlib
import pandas as pd

In [18]:
project_root = Path.cwd().parent

metadata_dir = (
    project_root
    / "data"
    / "metadata"
)

files = [
    "svm_closed_set_enrollment.csv",
    "svm_closed_set_train.csv",
    "svm_closed_set_validation.csv",
    "svm_closed_set_test.csv",
    "cosine_validation_enrollment.csv",
    "cosine_validation_query.csv",
    "cosine_validation_unknown.csv",
    "cosine_test_enrollment.csv",
    "cosine_test_query.csv",
    "cosine_test_unknown.csv"
]

dfs = []

for f in files:
    dfs.append(
        pd.read_csv(metadata_dir / f)
    )

all_audio = pd.concat(
    dfs,
    ignore_index=True
)

### Kiểm tra duplicate path

In [19]:
duplicate_path = (
    all_audio[
        all_audio.duplicated(
            subset="audio_path",
            keep=False
        )
    ]
    .sort_values("audio_path")
)

print(len(duplicate_path))

0


### Kiểm tra duplicate checksum

In [20]:
print(all_audio["checksum"].head())
print(all_audio["checksum"].isna().sum())
print(len(all_audio))

0    54dd762665a7b4b1ab3c79292e78022bdaff86cb8b8c5f...
1    b48193af5e64eab3da8fef115833f4165598a0219e4fd4...
2    3f50433277c1b044058d5520ff9d09b89560ab054f86c1...
3    c22c7d304ae743e31e158618be88002e96c2b4415e0c0b...
4    d3fbfbf010c3c65e711f0938b6b61a881217193debf74e...
Name: checksum, dtype: object
0
550


In [21]:
duplicate_checksum = (
    all_audio[
        all_audio.duplicated(
            subset="checksum",
            keep=False
        )
    ]
    .sort_values("checksum")
)

print(len(duplicate_checksum))

0


### Sinh và lưu report

In [22]:
duplicate_report = all_audio.copy()

duplicate_report["duplicate_path"] = duplicate_report.duplicated(
    subset="audio_path",
    keep=False
)

duplicate_report["duplicate_checksum"] = duplicate_report.duplicated(
    subset="checksum",
    keep=False
)

duplicate_report.to_csv(
    metadata_dir / "duplicate_report.csv",
    index=False,
    encoding="utf-8-sig"
)

## PHẦN 2: Leakage checking

In [23]:
splits = {}

for f in files:

    splits[f] = set(
        pd.read_csv(
            metadata_dir / f
        )["audio_id"]
    )

In [24]:
from itertools import combinations

leakage = []

for a, b in combinations(
    splits.keys(),
    2
):

    overlap = splits[a] & splits[b]

    leakage.append({

        "split_1": a,
        "split_2": b,
        "num_overlap": len(overlap),
        "audio_ids": list(overlap)
    })

leakage = pd.DataFrame(leakage)

In [25]:
print(
    leakage["num_overlap"].sum()
)

0


### Sinh report

In [27]:
report = []
report.append("# Leakage Report\n")

for _, row in leakage.iterrows():
    report.append(
        f"## {row.split_1} vs {row.split_2}"
    )
    report.append(
        f"- Overlap: {row.num_overlap}"
    )
    report.append("")

with open(
    metadata_dir / "leakage_report.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report)
    )

## PHẦN 3. Speaker leakage

In [28]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

metadata_dir = (
    project_root
    / "data"
    / "metadata"
)


svm_train = pd.read_csv(
    metadata_dir / "svm_closed_set_train.csv"
)

svm_validation = pd.read_csv(
    metadata_dir / "svm_closed_set_validation.csv"
)

svm_test = pd.read_csv(
    metadata_dir / "svm_closed_set_test.csv"
)


cosine_validation_enrollment = pd.read_csv(
    metadata_dir / "cosine_validation_enrollment.csv"
)

cosine_validation_query = pd.read_csv(
    metadata_dir / "cosine_validation_query.csv"
)

cosine_validation_unknown = pd.read_csv(
    metadata_dir / "cosine_validation_unknown.csv"
)


cosine_test_enrollment = pd.read_csv(
    metadata_dir / "cosine_test_enrollment.csv"
)

cosine_test_query = pd.read_csv(
    metadata_dir / "cosine_test_query.csv"
)

cosine_test_unknown = pd.read_csv(
    metadata_dir / "cosine_test_unknown.csv"
)

In [30]:
svm_experimental_speakers = set(
    pd.concat(
        [
            svm_train,
            svm_validation,
            svm_test
        ]
    )["speaker_id"]
)

cosine_validation_speakers = set(
    pd.concat(
        [
            cosine_validation_enrollment,
            cosine_validation_query,
            cosine_validation_unknown
        ]
    )["speaker_id"]
)

cosine_test_speakers = set(
    pd.concat(
        [
            cosine_test_enrollment,
            cosine_test_query,
            cosine_test_unknown
        ]
    )["speaker_id"]
)

validation_enrolled_speakers = set(
    cosine_validation_enrollment["speaker_id"]
)

validation_unknown_speakers = set(
    cosine_validation_unknown["speaker_id"]
)

test_enrolled_speakers = set(
    cosine_test_enrollment["speaker_id"]
)

test_unknown_speakers = set(
    cosine_test_unknown["speaker_id"]
)

In [31]:
def check_speaker_leakage(
    speakers_a,
    speakers_b
):
    return speakers_a & speakers_b

In [32]:
speaker_leakage_results = {

    "SVM experimental speakers vs Cosine validation speakers":
        check_speaker_leakage(
            svm_experimental_speakers,
            cosine_validation_speakers
        ),

    "SVM experimental speakers vs Cosine test speakers":
        check_speaker_leakage(
            svm_experimental_speakers,
            cosine_test_speakers
        ),

    "Cosine validation speakers vs Cosine test speakers":
        check_speaker_leakage(
            cosine_validation_speakers,
            cosine_test_speakers
        ),

    "Validation enrolled speakers vs Validation unknown speakers":
        check_speaker_leakage(
            validation_enrolled_speakers,
            validation_unknown_speakers
        ),

    "Test enrolled speakers vs Test unknown speakers":
        check_speaker_leakage(
            test_enrolled_speakers,
            test_unknown_speakers
        )
}

In [33]:
for name, overlap in speaker_leakage_results.items():
    print(name)
    if len(overlap) == 0:
        print("PASS: set()")
    else:
        print("FAIL:", overlap)
    print()

SVM experimental speakers vs Cosine validation speakers
PASS: set()

SVM experimental speakers vs Cosine test speakers
PASS: set()

Cosine validation speakers vs Cosine test speakers
PASS: set()

Validation enrolled speakers vs Validation unknown speakers
PASS: set()

Test enrolled speakers vs Test unknown speakers
PASS: set()



In [34]:
speaker_leakage_report = pd.DataFrame(
    [
        {
            "check": name,
            "overlap_speaker":

                ",".join(sorted(list(overlap)))
                if overlap
                else "",

            "status":

                "PASS"
                if len(overlap) == 0
                else "FAIL"
        }

        for name, overlap
        in speaker_leakage_results.items()
    ]
)

speaker_leakage_report.to_csv(
    metadata_dir /
    "speaker_leakage_report.csv",

    index=False,

    encoding="utf-8-sig"
)
speaker_leakage_report

,check,overlap_speaker,status
0,SVM experimental speakers vs Cosine validation...,,PASS
1,SVM experimental speakers vs Cosine test speakers,,PASS
2,Cosine validation speakers vs Cosine test spea...,,PASS
3,Validation enrolled speakers vs Validation unk...,,PASS
4,Test enrolled speakers vs Test unknown speakers,,PASS
